In [5]:
"""
Value Iteration on a 5x5 GridWorld (from scratch, pure NumPy)
No external RL libraries used - self contained so it always runs.
"""
import numpy as np

# ---------- 1. Define the GridWorld ----------
GRID_SIZE = 5
GAMMA = 0.9          # discount factor
THETA = 1e-4         # convergence threshold

In [6]:
# Grid legend:  S = start, G = goal (+1 reward), H = hole (-1 reward, terminal), . = normal cell (-0.04 step cost)
grid_layout = [
    ['S', '.', '.', '.', '.'],
    ['.', 'H', '.', 'H', '.'],
    ['.', '.', '.', '.', '.'],
    ['.', 'H', '.', '.', '.'],
    ['.', '.', '.', 'H', 'G'],
]

ACTIONS = ['UP', 'DOWN', 'LEFT', 'RIGHT']
ACTION_DELTA = {
    'UP': (-1, 0),
    'DOWN': (1, 0),
    'LEFT': (0, -1),
    'RIGHT': (0, 1),
}


In [7]:
def is_terminal(r, c):
    return grid_layout[r][c] in ('H', 'G')


In [8]:
def reward(r, c):
    cell = grid_layout[r][c]
    if cell == 'G':
        return 1.0
    if cell == 'H':
        return -1.0
    return -0.04  # small step cost so the agent prefers shorter paths

In [9]:
def next_state(r, c, action):
    """Deterministic transition. Bumping into a wall keeps the agent in place."""
    if is_terminal(r, c):
        return r, c
    dr, dc = ACTION_DELTA[action]
    nr, nc = r + dr, c + dc
    if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
        return nr, nc
    return r, c  # wall bump

In [10]:
# ---------- 2. Value Iteration ----------
def value_iteration():
    V = np.zeros((GRID_SIZE, GRID_SIZE))
    iteration = 0
    while True:
        delta = 0.0
        new_V = V.copy()
        for r in range(GRID_SIZE):
            for c in range(GRID_SIZE):
                if is_terminal(r, c):
                    new_V[r, c] = reward(r, c)
                    continue
                action_values = []
                for a in ACTIONS:
                    nr, nc = next_state(r, c, a)
                    q_sa = reward(r, c) + GAMMA * V[nr, nc]
                    action_values.append(q_sa)
                new_V[r, c] = max(action_values)
                delta = max(delta, abs(new_V[r, c] - V[r, c]))
        V = new_V
        iteration += 1
        if delta < THETA:
            break
    return V, iteration

In [11]:
def extract_policy(V):
    policy = np.full((GRID_SIZE, GRID_SIZE), ' ', dtype='<U5')
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if is_terminal(r, c):
                policy[r, c] = grid_layout[r][c]
                continue
            best_a, best_val = None, -np.inf
            for a in ACTIONS:
                nr, nc = next_state(r, c, a)
                q_sa = reward(r, c) + GAMMA * V[nr, nc]
                if q_sa > best_val:
                    best_val = q_sa
                    best_a = a
            policy[r, c] = best_a
    return policy

In [12]:
if __name__ == "__main__":
    V, num_iterations = value_iteration()
    policy = extract_policy(V)

    print(f"Converged in {num_iterations} iterations\n")
    print("Final Value Table (rounded to 2 decimals):")
    print(np.round(V, 2))
    print("\nExtracted Optimal Policy:")
    for row in policy:
        print(' '.join(f"{cell:>5}" for cell in row))

Converged in 10 iterations

Final Value Table (rounded to 2 decimals):
[[ 0.2   0.27  0.34  0.43  0.52]
 [ 0.27 -1.    0.43 -1.    0.62]
 [ 0.34  0.43  0.52  0.62  0.73]
 [ 0.27 -1.    0.62  0.73  0.86]
 [ 0.34  0.43  0.52 -1.    1.  ]]

Extracted Optimal Policy:
 DOWN RIGHT  DOWN RIGHT  DOWN
 DOWN     H  DOWN     H  DOWN
RIGHT RIGHT  DOWN  DOWN  DOWN
   UP     H RIGHT RIGHT  DOWN
RIGHT RIGHT    UP     H     G


In [13]:
# Import numpy for array/matrix operations (used for the Value table and Policy grid)
import numpy as np

# ---- Environment Configuration ----
GRID_SIZE = 5      # The grid is 5x5
GAMMA = 0.9         # Discount factor: how much future rewards matter vs immediate ones
THETA = 1e-4        # Convergence threshold: stop policy evaluation when value changes are this small

In [14]:
# Define the GridWorld layout:
# 'S' = Start, '.' = normal walkable cell, 'H' = Hole (bad terminal state), 'G' = Goal (good terminal state)
grid_layout = [
    ['S', '.', '.', '.', '.'],
    ['.', 'H', '.', 'H', '.'],
    ['.', '.', '.', '.', '.'],
    ['.', 'H', '.', '.', '.'],
    ['.', '.', '.', 'H', 'G'],
]

# The 4 possible actions an agent can take in any state
ACTIONS = ['UP', 'DOWN', 'LEFT', 'RIGHT']

# Maps each action to a (row_change, col_change) so we know how to move on the grid
ACTION_DELTA = {'UP': (-1, 0), 'DOWN': (1, 0), 'LEFT': (0, -1), 'RIGHT': (0, 1)}

In [15]:
def is_terminal(r, c):
    """Returns True if the cell is a Hole or Goal (i.e., episode ends here)."""
    return grid_layout[r][c] in ('H', 'G')

def reward(r, c):
    """
    Returns the reward for being in cell (r, c):
    - Goal    -> +1.0 (big positive reward)
    - Hole    -> -1.0 (big penalty)
    - Normal  -> -0.04 (small penalty per step, encourages shortest path)
    """
    cell = grid_layout[r][c]
    if cell == 'G':
        return 1.0
    if cell == 'H':
        return -1.0
    return -0.04

def next_state(r, c, action):
    """
    Given current position (r, c) and an action, returns the resulting position.
    - If already in a terminal state, the agent stays put (no more movement).
    - If the action would move off the grid, the agent stays in the same cell (wall bump).
    """
    if is_terminal(r, c):
        return r, c
    dr, dc = ACTION_DELTA[action]
    nr, nc = r + dr, c + dc
    if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
        return nr, nc
    return r, c  # bumped into a wall, no movement

In [16]:
def policy_evaluation(policy, V):
    """
    Given a FIXED policy, repeatedly update the value table V until it stabilizes
    (i.e., until the biggest change across all cells is smaller than THETA).
    
    This answers: "If I always follow this exact policy, what's the expected
    long-term value of being in each cell?"
    """
    eval_sweeps = 0  # counts how many full passes over the grid were needed
    while True:
        delta = 0.0              # tracks the biggest change in this sweep
        new_V = V.copy()         # we compute new values based on OLD values (synchronous update)
        
        for r in range(GRID_SIZE):
            for c in range(GRID_SIZE):
                if is_terminal(r, c):
                    # Terminal states just hold their reward value, no lookahead needed
                    new_V[r, c] = reward(r, c)
                    continue
                
                # Follow whatever action the current policy says for this cell
                a = policy[r, c]
                nr, nc = next_state(r, c, a)
                
                # Bellman equation for a FIXED policy (no max over actions here,
                # since we're evaluating, not improving)
                new_V[r, c] = reward(r, c) + GAMMA * V[nr, nc]
                
                # Track the largest change to check for convergence
                delta = max(delta, abs(new_V[r, c] - V[r, c]))
        
        V = new_V
        eval_sweeps += 1
        
        if delta < THETA:   # values have stabilized, stop looping
            break
            
    return V, eval_sweeps

In [17]:
def policy_improvement(V):
    """
    Given a value table V, greedily choose the BEST action for every state.
    This is the "improvement" step: for each cell, look at all 4 possible actions,
    calculate what value each would lead to, and pick the action with the highest value.
    """
    new_policy = np.full((GRID_SIZE, GRID_SIZE), 'UP', dtype='<U5')  # placeholder policy
    
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if is_terminal(r, c):
                # Terminal cells don't need an action; just label them H or G for display
                new_policy[r, c] = grid_layout[r][c]
                continue
            
            best_a, best_val = None, -np.inf
            
            # Try every action and see which leads to the highest expected value
            for a in ACTIONS:
                nr, nc = next_state(r, c, a)
                q_sa = reward(r, c) + GAMMA * V[nr, nc]  # this is essentially Q(s,a)
                if q_sa > best_val:
                    best_val = q_sa
                    best_a = a
            
            new_policy[r, c] = best_a  # store the best action found
            
    return new_policy

In [18]:
def policy_iteration():
    """
    Full Policy Iteration algorithm:
    1. Start with an arbitrary policy (here: always move RIGHT)
    2. Evaluate it fully (Policy Evaluation)
    3. Improve it greedily (Policy Improvement)
    4. Repeat steps 2-3 until the policy stops changing (i.e., it's optimal)
    """
    V = np.zeros((GRID_SIZE, GRID_SIZE))  # start with all zero values
    
    # Initialize with an arbitrary starting policy: always move RIGHT
    policy = np.full((GRID_SIZE, GRID_SIZE), 'RIGHT', dtype='<U5')
    
    # Terminal cells get labeled with their symbol instead of an action
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if is_terminal(r, c):
                policy[r, c] = grid_layout[r][c]

    policy_improvement_steps = 0   # counts how many improve-evaluate rounds happened
    total_eval_sweeps = 0          # counts total inner sweeps across ALL evaluation calls

    while True:
        # Step 1: Evaluate current policy fully
        V, sweeps = policy_evaluation(policy, V)
        total_eval_sweeps += sweeps
        
        # Step 2: Improve policy greedily based on new values
        new_policy = policy_improvement(V)
        policy_improvement_steps += 1
        
        # Step 3: If policy didn't change at all, we've converged to the optimal policy
        if np.array_equal(new_policy, policy):
            break
        
        policy = new_policy  # otherwise, adopt the improved policy and repeat

    return V, policy, policy_improvement_steps, total_eval_sweeps

In [19]:
# Run Policy Iteration and unpack results
V, policy, improve_steps, eval_sweeps = policy_iteration()

# Print how many rounds it took to converge
print(f"Policy Iteration converged after {improve_steps} policy-improvement rounds")
print(f"(using {eval_sweeps} total policy-evaluation sweeps internally)\n")

# Print the final value table (how "good" each cell is under the optimal policy)
print("Final Value Table (rounded to 2 decimals):")
print(np.round(V, 2))

# Print the final optimal policy (best action to take in each cell)
print("\nFinal Optimal Policy:")
for row in policy:
    print(' '.join(f"{cell:>5}" for cell in row))

Policy Iteration converged after 8 policy-improvement rounds
(using 132 total policy-evaluation sweeps internally)

Final Value Table (rounded to 2 decimals):
[[ 0.2   0.27  0.34  0.43  0.52]
 [ 0.27 -1.    0.43 -1.    0.62]
 [ 0.34  0.43  0.52  0.62  0.73]
 [ 0.27 -1.    0.62  0.73  0.86]
 [ 0.34  0.43  0.52 -1.    1.  ]]

Final Optimal Policy:
 DOWN RIGHT  DOWN RIGHT  DOWN
 DOWN     H  DOWN     H  DOWN
RIGHT RIGHT  DOWN  DOWN  DOWN
   UP     H RIGHT RIGHT  DOWN
RIGHT RIGHT    UP     H     G
